In [ ]:
import pandas as pd
import matplotlib as plt
import numpy as np
import json, lz4.frame
import sys
import os
from pathlib import Path  
PATH = "C:\\Users\\omran\\code\\PokemonRL\\data\\data\\gen9ou\\2022\\11\\" 

In [12]:
# view one example
dir = PATH + "smogtours-gen9ou-663321_Unrated_pluck78484_vs_drifloon53330_11-30-2022_WIN.json.lz4"
with lz4.frame.open(dir, "rb") as f:
    data = json.loads(f.read().decode("utf-8"))
states, actions = data["states"], data["actions"]

In [14]:
actions

[3, 8, 2, 5, 8, 3, 6, 5, 2, 2, 5, 3, 0, 0, 4, 0, 9, 0, 4, 5, 0, 1, 5, 0, 0, -1]

In [15]:
numOfMoves = len(states)
print(len(actions))

26


In [35]:
states[2]


{'format': 'gen9ou',
 'player_active_pokemon': {'name': 'tinglu',
  'hp_pct': 0.7879377431906615,
  'types': 'dark ground',
  'item': 'leftovers',
  'ability': 'vesselofruin',
  'lvl': 100,
  'status': 'nostatus',
  'effect': 'noeffect',
  'moves': [{'name': 'stealthrock',
    'move_type': 'rock',
    'category': 'status',
    'base_power': 0,
    'accuracy': 1.0,
    'priority': 0,
    'current_pp': 31,
    'max_pp': 32},
   {'name': 'whirlwind',
    'move_type': 'normal',
    'category': 'status',
    'base_power': 0,
    'accuracy': 1.0,
    'priority': -6,
    'current_pp': 32,
    'max_pp': 32},
   {'name': 'earthquake',
    'move_type': 'ground',
    'category': 'physical',
    'base_power': 100,
    'accuracy': 1.0,
    'priority': 0,
    'current_pp': 16,
    'max_pp': 16},
   {'name': 'spikes',
    'move_type': 'ground',
    'category': 'status',
    'base_power': 0,
    'accuracy': 1.0,
    'priority': 0,
    'current_pp': 32,
    'max_pp': 32}],
  'atk_boost': 0,
  'spa_boos

Actions and states link up <br>
0–3: use the active Pokémon's moves, but in alphabetical order by move name — not JSON list order, not PS slot order <br>
4–8: switch to the other party members, also alphabetically <br>
9–12: gen-specific gimmick wildcards — for Gen 9 this means "use a move with terastallize," moves again taken alphabetically

In [ ]:
#check how many unique values there are and check them against the vocabulary list

def collect_strings_dfs(data: dict | list) -> set[str]:
    strings_set = set()

    def _dfs(node):
        if isinstance(node, dict):
            for key, value in node.items():
                # If you also want dictionary keys, uncomment the line below:
                # if isinstance(key, str): strings_set.add(key)
                _dfs(value)
        elif isinstance(node, list):
            for item in node:
                _dfs(item)
        elif isinstance(node, str):
            temp = node.split(" ")
            for s in temp:
                strings_set.add(s)

    _dfs(data)
    return strings_set


# result = collect_strings_dfs(states[0])
# print(result)

{'shadowball', 'iceshard', 'spikes', 'gen9ou', 'unknownitem', 'mortalspin', 'steel', 'choicescarf', 'crunch', 'vesselofruin', 'notype', 'thunderbolt', 'levitate', 'earthquake', 'noconditions', 'sludgewave', 'whirlwind', 'normal', 'rock', 'electric', 'ice', 'ghost', 'sandyshocks', 'grass', 'nomove', 'ironvaliant', 'chiyu', 'corviknight', 'dark', 'protect', 'toxicdebris', 'ground', 'energyball', 'noeffect', 'leftovers', 'stealthrock', 'heavydutyboots', 'special', 'nostatus', 'protosynthesis', 'quarkdrive', 'voltswitch', 'slitherwing', 'fairy', 'glimmora', 'closecombat', 'goodasgold', 'status', 'fighting', 'rotom', 'chienpao', 'knockoff', 'gholdengo', 'focussash', 'poison', 'water', 'tinglu', 'boosterenergy', 'physical', 'swordofruin', 'makeitrain', 'nofield', 'trick', 'psychic', 'noweather', 'iciclecrash', 'rotomwash', 'dazzlinggleam', 'torkoal', 'swordsdance', 'hydropump'}


In [ ]:
with open("DefaultObservationSpace-v1.json", "r") as file:
    vocab = json.load(file)
result = set(collect_strings_dfs(states[0]))
count =0
for wrd in result:
    try:
        vocab.get(wrd)
        count +=1
    except:
        print(f"skipping word {wrd} bc not in vocba")


print("\n")
print(f"Total words in vocab percentage: {count/len(result)}")





Total words in vocab percentage: 1.0


In [ ]:
# random sampling for broader representation
import random

#use random walk down a path for much more quicker random file access

def get_random_file_via_walk(root_path: Path, max_attempts: int = 100) -> Path | None:
    """Walks randomly down the directory tree until it hits a file."""
    curr = root_path
    attempts = 0
    
    while attempts < max_attempts:
        attempts += 1
        if not curr.is_dir():
            return curr  # found a file
        
        try:
            # read direct children of the current directory
            children = list(curr.iterdir())
        except PermissionError:
            return None
            
        if not children:
            # Empty folder, back up to root or pick another path
            curr = root_path
            continue
            
        #  random subfolder or file
        curr = random.choice(children)
        
    return curr if curr.is_file() else None

def fast_sample_files(root_dir: str, sample_size: int = 500) -> list[Path]:
    root = Path(root_dir)
    sampled_files = set()
    
    while len(sampled_files) < sample_size:
        selected = get_random_file_via_walk(root)
        if selected and selected.is_file():
            sampled_files.add(selected)
            
    return list(sampled_files)
data_dir = "C:\\Users\\omran\\code\\PokemonRL\\data\\data\\gen9ou\\"
files = fast_sample_files(data_dir, sample_size=500)

In [ ]:
count = 0
totalWords = 0
#load in vocab to ensure I have all possible words and check if anything is missing
with open("DefaultObservationSpace-v1.json", "r") as file:
    vocab = json.load(file)

for i, f in enumerate(files):
    with lz4.frame.open(f, "rb") as filename: #load the file
        data = json.loads(filename.read().decode("utf-8"))
    
    states = data["states"]
    #get the words
    wrds = set(collect_strings_dfs(states[0]))
    for wrd in wrds:
        try:
            vocab.get(wrd)#if its in the vocab, add to the counter
            count +=1
        except:
            print(f"skipping word {wrd} bc not in vocab")
    
    totalWords += len(wrds)

    print(f"Completed {i}th file!")

print("="*50)
print(print(f"Total words in vocab percentage: {count/totalWords}"))
# output was 1.0, so we should have ALL the words


Completed 0th file!
Completed 1th file!
Completed 2th file!
Completed 3th file!
Completed 4th file!
Completed 5th file!
Completed 6th file!
Completed 7th file!
Completed 8th file!
Completed 9th file!
Completed 10th file!
Completed 11th file!
Completed 12th file!
Completed 13th file!
Completed 14th file!
Completed 15th file!
Completed 16th file!
Completed 17th file!
Completed 18th file!
Completed 19th file!
Completed 20th file!
Completed 21th file!
Completed 22th file!
Completed 23th file!
Completed 24th file!
Completed 25th file!
Completed 26th file!
Completed 27th file!
Completed 28th file!
Completed 29th file!
Completed 30th file!
Completed 31th file!
Completed 32th file!
Completed 33th file!
Completed 34th file!
Completed 35th file!
Completed 36th file!
Completed 37th file!
Completed 38th file!
Completed 39th file!
Completed 40th file!
Completed 41th file!
Completed 42th file!
Completed 43th file!
Completed 44th file!
Completed 45th file!
Completed 46th file!
Completed 47th file!
Co